#### Dockerfile creation
we first create a Dockerfile based on the documentation here https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-jl-admin-guide-custom-images.html#studio-updated-jl-custom-images-dockerfile-templates

```
FROM public.ecr.aws/amazonlinux/amazonlinux:2023

ARG NB_USER="sagemaker-user"
ARG NB_UID=1000
ARG NB_GID=100

# Install Python3, pip, and other dependencies
RUN yum install -y \
    python3 \
    python3-pip \
    python3-devel \
    gcc \
    shadow-utils && \
    useradd --create-home --shell /bin/bash --gid "${NB_GID}" --uid ${NB_UID} ${NB_USER} && \
    yum clean all

RUN python3 -m pip install --no-cache-dir \
    'jupyterlab>=4.0.0,<5.0.0' \
    urllib3 \
    jupyter-activity-monitor-extension \
    --ignore-installed

# Verify versions
RUN python3 --version && \
    jupyter lab --version

USER ${NB_UID}
CMD jupyter lab --ip 0.0.0.0 --port 8888 \
    --ServerApp.base_url="/jupyterlab/default" \
    --ServerApp.token='' \
    --ServerApp.allow_origin='*'
```

In [ ]:
# we create the ECR repository
!aws ecr create-repository --repository-name smstudio-custom --region us-east-1

In [ ]:
!aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 609009159737.dkr.ecr.us-east-1.amazonaws.com

In [ ]:
!pip install sagemaker-studio-image-build

In [ ]:
!aws sts get-caller-identity

make sure to give the proper permissinos and trust policy. An example trust policy is ```{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AssumeRole",
            "Effect": "Allow",
            "Principal": {
                "Service": [
                    "sagemaker.amazonaws.com",
                    "codebuild.amazonaws.com"
                ]
            },
            "Action": "sts:AssumeRole"
        }
    ]
}```

In [ ]:
!sm-docker build -t smstudio-custom:my-jupyterlab-image .

In [ ]:
# 609009159737.dkr.ecr.us-east-1.amazonaws.com/sagemaker-studio:latest

# Create an AppImageConfig:

```
{
  "AppImageConfigName": "my-custom-image-config",
    "JupyterLabAppImageConfig": { 
        "FileSystemConfig": { 
            "MountPath": "/home/sagemaker-user",
            "DefaultUid": 1000,
            "DefaultGid": 100
      }
   }
}
```

In [ ]:
!aws sagemaker create-app-image-config --cli-input-json file://app-image-config-input.json --region us-east-1

In [ ]:
# arn:aws:sagemaker:us-east-1:609009159737:app-image-config/my-custom-image-config

In [ ]:
!aws sagemaker create-image \
  --image-name my-jupyterlab-image2 \
  --role-arn arn:aws:iam::609009159737:role/service-role/SageMaker-ExecutionRole-20250723T112596 \
  --region us-east-1

In [ ]:
# arn:aws:sagemaker:us-east-1:609009159737:image/my-jupyterlab-image

In [ ]:
!aws sagemaker create-image-version \
  --image-name my-jupyterlab-image2 \
  --base-image 609009159737.dkr.ecr.us-east-1.amazonaws.com/sagemaker-studio:latest \
  --region us-east-1

In [ ]:
# arn:aws:sagemaker:us-east-1:794038231401:image-version/my-jupyterlab-image/1

In [ ]:
!aws sagemaker update-domain \
  --domain-id d-hbzcqghviqax \
  --default-space-settings file://default-space-settings.json \
  --region us-east-1

In [ ]:
!aws sagemaker describe-domain --domain-id d-hbzcqghviqax